In [1]:
from copy import deepcopy
import gymnasium as gym
from gymnasium import spaces
from gymnasium.envs.registration import register
import ray
import numpy as np
import json
import scipy.stats as st
import matplotlib.pyplot as plt

import random
import string
import pprint
from pathlib import Path

import tensorflow as tf
import argparse
import numpy as np
from ray.rllib.algorithms.algorithm import Algorithm
from numpy import inf
import scipy.integrate as sigr
import ray.rllib.algorithms.apo as apo
import ray.rllib.algorithms.ppo as ppo 
from ray import air, tune
from ray.tune.registry import register_env, register_trainable
from ray.tune.schedulers import PopulationBasedTraining
from ray.tune import ResultGrid

from ray.rllib.utils import try_import_tf
from ray.rllib.utils.schedules.polynomial_schedule import PolynomialSchedule
from ray.tune.logger import pretty_print
from ray.air.config import CheckpointConfig
from CheeseEnvironment import CheeseEnv as env
from CheeseEnvironment import upper_bound as ub_function
#change to multiagent for multiagent environment
from MultiAgentCheeseEnvironment import CheeseEnv as multi_env
#from MultiAgentCheeseEnvironment import upper_bound as ub_function
from ray.rllib.algorithms.apo.apo_tf_policy import APOTF2Policy
from ray.rllib.utils.checkpoints import get_checkpoint_info
from stochastic.processes.diffusion import vasicek as vsk

import wandb
import openpyxl
import os

c:\Users\ga84cib\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
pids = ["cheese_010", "cheese_012", "cheese_020", "cheese_022", "cheese_110", "cheese_112", "cheese_120", "cheese_122", "cheese_210", "cheese_212", "cheese_220", "cheese_222", "cheese_310", "cheese_312", "cheese_320", "cheese_322"] #"cheese_210", "cheese_220", "cheese_212", "cheese_222", "cheese_310", "cheese_320", "cheese_312", "cheese_322"] 

for pid in pids:

    print(f"Starting evaluation for problem {pid}")
    problem_id = pid

    no_correl_id = problem_id[:8] + "nn"

    id_path = f"{os.getcwd()}/cheese/{no_correl_id}/training_results/full_blending_py_nn_x_lp_100w"

    problem_path = f"{os.getcwd()}/cheese/{problem_id}"

    path_config = f"{os.getcwd()}/cheese/{problem_id}/config.json"

    with open(path_config, 'r') as f:
            problem_config = json.load(f)
    numAges = problem_config["numAges"]
    nProducts = problem_config["nProducts"]
    targetAges = problem_config["targetAges"]
    ageRange = problem_config["ageRange"]
    maxInventory = problem_config["maxInventory"]
    evaporation = problem_config["evaporation"]
    demand_elasticity = problem_config["demand_elasticity"]
    demand_means = problem_config["demand_means"]
    demand_covs = problem_config["demand_covs"]
    salvage = problem_config["salvage"]
    price_means = problem_config["price_means"]
    speed = problem_config["speed"]
    vol = problem_config["vol"]
    price_covs = problem_config["price_covs"]
    correlation_matrix = problem_config["correlation_matrix"]
    holding_cost = problem_config["holdingCosts"]
    production_step_size = problem_config["production_step_size"]
    demand_max = problem_config["demand_max"]
    min_ppf = problem_config["min_ppf"]
    max_ppf = problem_config["max_ppf"]
    expected_revenue = {p: {float(l): {float(g): problem_config["expected_revenue"][str(p)][str(l)][str(g)] for g in problem_config["expected_revenue"][str(p)][str(l)]} for l in problem_config["expected_revenue"][str(p)]} for p in range(nProducts)}
    slope = {p: {float(l): {float(g): problem_config["slope"][str(p)][str(l)][str(g)] for g in problem_config["slope"][str(p)][str(l)]} for l in problem_config["slope"][str(p)]} for p in range(nProducts)}
    upper_bound = problem_config["upper_bound"] if "upper_bound" in problem_config else None
    priceDistributions = [st.norm(price_means[i], price_covs[i]*price_means[i]) for i in range(nProducts+1)]
    price_step_size = 0.1
    production_step_size_lp = 0.1   

    #determine use of linear program for issuance decisions
    use_issuance_model = True

    #allow blending
    allow_blending = True
    blending_range = None

    #multi agent setting
    multi_agent_setting = None #"multi_group"

    #create config dict which is passed to environment init 
    AIE_config = {"numAges":numAges, "nProducts":nProducts, "targetAges":targetAges, "ageRange":ageRange, "maxInventory":maxInventory, "evaporation":evaporation, 
                "demand_elasticity": demand_elasticity, "demand_means":demand_means, "demand_covs":demand_covs, "salvage":salvage, "price_means": price_means, "speed": speed,
                "vol": vol, "price_covs": price_covs, "correlation_matrix": correlation_matrix, "priceDistributions":priceDistributions, "holdingCosts":holding_cost, "expected_revenue":expected_revenue, "slope": slope,
                "min_ppf":min_ppf, "max_ppf":max_ppf, "production_step_size":production_step_size, "production_step_size_lp":production_step_size_lp, "price_step_size":price_step_size, "upper_bound":upper_bound, "demand_max": demand_max,
                "action_space_design":"box_continuous", "render_mode":'rgb_array', "horizon":5500, "simulate_heuristic":False, "use_common_random_numbers": False,
                "reward_lb":-1.0, "reward_ub":1.0, "use_issuance_model":use_issuance_model, "allow_blending":allow_blending, "blending_range":blending_range, "multi_agent_setting":multi_agent_setting} 
    
    #read the best checkpoint
    with open(f"{id_path}/best_checkpoint.json") as f:
        best_checkpoint = json.load(f)

    used_checkpoint = best_checkpoint["eval"][-1][1]
    data_size = 50_000

    ray.shutdown()
    ray.init()

    register_env("CheeseEnvironment", lambda config: env(config))
    test_env = env(AIE_config)

    if AIE_config["simulate_heuristic"] is False:
        checkpoint_info = get_checkpoint_info(used_checkpoint)
        print("Checkpoint info:", checkpoint_info)
        #raise ValueError(f"Checkpoint info: {checkpoint_info}")
        state = Algorithm._checkpoint_info_to_algorithm_state(
            checkpoint_info = checkpoint_info,
            policy_ids = None,
            policy_mapping_fn=None,
            policies_to_train=None,
        )
        state["config"]["num_workers"] = 1
        policy = Algorithm.from_state(state)
        print("Policy loaded")
        test_env = env(AIE_config)

        # features, responses, rewards = test_env.simulate_data_for_regression(policy, data_size=data_size)
        # regression_data = {"features":features.tolist(), "responses":responses.tolist(), "rewards": rewards.tolist()}
        # with open(id_path+"/regression_data.json", 'w') as f:
        #     json.dump(regression_data, f)

        #load cdfs for evaluation
    else:
         policy = None
    with open("cheese/evaluation_data_aligned.json") as f:
        cdfs = json.load(f)[str(correlation_matrix[0][1])]
    features, responses, rewards = test_env.simulate_w_cdfs(cdfs, policy, warm_up_length=1000)
    regression_data = {"features":features.tolist(), "responses":responses.tolist(), "rewards": rewards.tolist()}
    with open(problem_path+"/regression_data_crn_nocorrel.json", 'w') as f:
        json.dump(regression_data, f)

Starting evaluation for problem cheese_010


2025-06-05 13:53:57,594	INFO worker.py:1553 -- Started a local Ray instance.


SALES BOUND:  [20.6, 14.4]
MAX REWARD:  1253.400429061677
MIN_REWARD:  0
starting inventory:  [17.71979916 17.71979916 17.71979916 17.71979916  8.50752798  8.50752798
  8.50752798  8.50752798  0.          0.        ]
Checkpoint info: {'type': 'Algorithm', 'checkpoint_version': <Version('1.0')>, 'checkpoint_dir': 'C:\\Users\\ga84cib\\Documents\\gitlab\\cheese_drl/cheese/cheese_0nn/training_results/full_blending_py_nn_x_lp_100w\\checkpoint_000481', 'state_file': 'C:\\Users\\ga84cib\\Documents\\gitlab\\cheese_drl/cheese/cheese_0nn/training_results/full_blending_py_nn_x_lp_100w\\checkpoint_000481\\algorithm_state.pkl', 'policy_ids': {'default_policy'}}


c:\Users\ga84cib\AppData\Local\Programs\Python\Python310\lib\site-packages\gymnasium\spaces\box.py:130: UserWarning: WARN: Box bound precision lowered by casting to float32
  gym.logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


C:\Users\ga84cib\Documents\gitlab\cheese_drl/cheese/cheese_0nn/training_results/full_blending_py_nn_x_lp_100w\checkpoint_000481\policies\default_policy\policy_state.pkl
(RolloutWorker pid=24008) SALES BOUND:  [20.6, 14.4]
(RolloutWorker pid=24008) MAX REWARD:  1253.4004290616951
(RolloutWorker pid=24008) MIN_REWARD:  0
(RolloutWorker pid=24008) Set parameter Username
(RolloutWorker pid=24008) Academic license - for non-commercial use only - expires 2025-09-17


...

(RolloutWorker pid=23008) starting inventory:  [16.91591904 16.91591904 16.91591904 16.91591904  8.47222538  8.47222538
(RolloutWorker pid=23008)   8.47222538  8.47222538  0.          0.        ]
(RolloutWorker pid=23008) FINISH EPISODE
(RolloutWorker pid=23008) FINISH EPISODE


(RolloutWorker pid=23008) c:\Users\ga84cib\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\core\fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
(RolloutWorker pid=23008)   return _methods._mean(a, axis=axis, dtype=dtype,
(RolloutWorker pid=23008) c:\Users\ga84cib\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\core\_methods.py:192: RuntimeWarning: invalid value encountered in divide
(RolloutWorker pid=23008)   ret = ret.dtype.type(ret / rcount)
2025-06-05 18:10:49,289	INFO trainable.py:172 -- Trainable.setup took 240.206 seconds. If your trainable is slow to initialize, consider setting reuse_actors=True to reduce actor creation overheads.
2025-06-05 18:10:49,325	WARNING util.py:67 -- Install gputil for GPU system monitoring.


Policy loaded
SALES BOUND:  [20.6, 14.4]


(RolloutWorker pid=23008) 2025-06-05 18:13:55,244	WARNING eager_tf_policy_v2.py:702 -- Cannot restore an optimizer's state for tf eager! Keras is not able to save the v1.x optimizers (from tf.compat.v1.train) since they aren't compatible with checkpoints.


MAX REWARD:  1378.1857090722824
MIN_REWARD:  0
starting inventory:  [16.91495672 16.91495672 16.91495672 16.91495672  8.4718191   8.4718191
  8.4718191   8.4718191   0.          0.        ]
CDF LENGTH:  51000
WARM-UP FINISHED
SIMULATING WITH CDFS... 1000 STEPS COMPLETED
SIMULATING WITH CDFS... 2000 STEPS COMPLETED
SIMULATING WITH CDFS... 3000 STEPS COMPLETED
SIMULATING WITH CDFS... 4000 STEPS COMPLETED
SIMULATING WITH CDFS... 5000 STEPS COMPLETED
SIMULATING WITH CDFS... 6000 STEPS COMPLETED
SIMULATING WITH CDFS... 7000 STEPS COMPLETED
SIMULATING WITH CDFS... 8000 STEPS COMPLETED
SIMULATING WITH CDFS... 9000 STEPS COMPLETED
SIMULATING WITH CDFS... 10000 STEPS COMPLETED
SIMULATING WITH CDFS... 11000 STEPS COMPLETED
SIMULATING WITH CDFS... 12000 STEPS COMPLETED
SIMULATING WITH CDFS... 13000 STEPS COMPLETED
SIMULATING WITH CDFS... 14000 STEPS COMPLETED
SIMULATING WITH CDFS... 15000 STEPS COMPLETED
SIMULATING WITH CDFS... 16000 STEPS COMPLETED
SIMULATING WITH CDFS... 17000 STEPS COMPLETED
S